# This python file will do operation on ALL FILES

In [ ]:
from google.colab import files
import os

# Create the local directory in Colab
os.makedirs('/content/15_Min', exist_ok=True)

print("Please upload your .xlsx files from your local '15_Min' folder:")
uploaded = files.upload()

# Move uploaded files to the directory
for filename in uploaded.keys():
    os.rename(filename, os.path.join('/content/15_Min', filename))

print("Upload complete.")

In [ ]:
# Updating directory paths to work correctly in Colab
import os

# Using /content/ for Google Colab environment
input_directory = '/content/15_Min/'
output_directory = '/content/Indicators/'
os.makedirs(output_directory, exist_ok=True)

# Update the analysis results path
output_excel_file = "/content/Analysis_of_Stocks.xlsx"
final_output_file = "/content/Final_Result.xlsx"

## Update the Search Date daily

In [ ]:
search_date = '2026-04-15'  # Today's Date

## Calculations & BUY-SELL Indications

In [ ]:
import pandas as pd
import numpy as np
import os
import time
from concurrent.futures import ThreadPoolExecutor

# Parameters for calculations
per = 50  # Sampling period
mult = 3.0  # Range multiplier

def smoothrng(df, period, multiplier):
    df['EMA'] = df['Close'].ewm(span=period, adjust=False).mean()
    df['AbsDiff'] = df['Close'].diff().abs()
    df['AvgRange'] = df['AbsDiff'].ewm(span=period, adjust=False).mean()
    df['SmoothRange'] = df['AvgRange'].ewm(span=(period * 2 - 1), adjust=False).mean() * multiplier
    return df

def rngfilt(df):
    df['Filter'] = df['Close'].copy()
    for i in range(1, len(df)):
        prev_filter = df.at[df.index[i - 1], 'Filter']
        smooth_range = df.at[df.index[i], 'SmoothRange']
        current_close = df.at[df.index[i], 'Close']
        if current_close > prev_filter:
            df.at[df.index[i], 'Filter'] = max(prev_filter, current_close - smooth_range)
        else:
            df.at[df.index[i], 'Filter'] = min(prev_filter, current_close + smooth_range)
    return df

def generate_signals(df):
    df['Upward'] = 0
    df['Downward'] = 0
    df['LongCondition'] = False
    df['ShortCondition'] = False
    df['PreviousCondition'] = 0
    df['Signal'] = np.nan
    for i in range(1, len(df)):
        if df.at[df.index[i], 'Filter'] > df.at[df.index[i - 1], 'Filter']:
            df.at[df.index[i], 'Upward'] = df.at[df.index[i - 1], 'Upward'] + 1
        else:
            df.at[df.index[i], 'Upward'] = 0
        if df.at[df.index[i], 'Filter'] < df.at[df.index[i - 1], 'Filter']:
            df.at[df.index[i], 'Downward'] = df.at[df.index[i - 1], 'Downward'] + 1
        else:
            df.at[df.index[i], 'Downward'] = 0
        df.at[df.index[i], 'LongCondition'] = (df.at[df.index[i], 'Close'] > df.at[df.index[i], 'Filter']) and (df.at[df.index[i], 'Upward'] > 0)
        df.at[df.index[i], 'ShortCondition'] = (df.at[df.index[i], 'Close'] < df.at[df.index[i], 'Filter']) and (df.at[df.index[i], 'Downward'] > 0)
        if df.at[df.index[i], 'LongCondition'] and df.at[df.index[i - 1], 'PreviousCondition'] == -1:
            df.at[df.index[i], 'Signal'] = 'Buy'
        elif df.at[df.index[i], 'ShortCondition'] and df.at[df.index[i - 1], 'PreviousCondition'] == 1:
            df.at[df.index[i], 'Signal'] = 'Sell'
        if df.at[df.index[i], 'LongCondition']:
            df.at[df.index[i], 'PreviousCondition'] = 1
        elif df.at[df.index[i], 'ShortCondition']:
            df.at[df.index[i], 'PreviousCondition'] = -1
        else:
            df.at[df.index[i], 'PreviousCondition'] = df.at[df.index[i - 1], 'PreviousCondition']
    return df

def calculate_rsi(df, period=14):
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period, min_periods=1).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period, min_periods=1).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    return df

def process_single_file(file):
    try:
        file_path = os.path.join(input_directory, file)
        df = pd.read_excel(file_path)
        df.columns = ['DateTime', 'Open', 'High', 'Low', 'Close', 'Volume', 'OI', 'RSI_Old']
        df['DateTime'] = pd.to_datetime(df['DateTime'])
        df.set_index('DateTime', inplace=True)
        df.sort_index(inplace=True)
        df = smoothrng(df, per, mult)
        df = rngfilt(df)
        df = calculate_rsi(df)
        df = generate_signals(df)
        signals = df[['Close', 'Filter', 'SmoothRange', 'RSI', 'Signal']].dropna(subset=['Signal'], how='all')
        output_file_path = os.path.join(output_directory, file)
        signals.to_excel(output_file_path)
        return f'Processed: {file}'
    except Exception as e:
        return f'Error in {file}: {e}'

files_to_process = [f for f in os.listdir(input_directory) if f.endswith('.xlsx')]
print(f'Starting parallel processing of {len(files_to_process)} files using threads...')

start_time = time.time()
with ThreadPoolExecutor() as executor:
    results = list(executor.map(process_single_file, files_to_process))
end_time = time.time()

for res in results:
    print(res)

print('-' * 30)
print(f'All files processed in {end_time - start_time:.2f} seconds.')

## Reverse Operations

In [ ]:
import pandas as pd
import os

# Using Colab path for reversed data
files = [f for f in os.listdir(output_directory) if f.endswith('.xlsx')]

for file_name in files:
    file_path = os.path.join(output_directory, file_name)

    # Load the Excel file
    df = pd.read_excel(file_path, engine='openpyxl')

    # Reverse order
    if len(df) > 1:
        df_reversed = pd.concat([df.head(0), df.tail(-1).iloc[::-1]], ignore_index=True)
    else:
        df_reversed = df

    # Write back to the same file
    df_reversed.to_excel(file_path, index=False, engine='openpyxl')
    print(f'Reversed: {file_name}')

print("All files reversed.")

## RSI Operations

In [ ]:
# import pandas as pd
# import os

# # Directory paths
# indicators_dir = r'C:\Users\adity\Desktop\ML-2\Stocks\Stocks\Indicators'
# fifteen_min_dir = r'C:\Users\adity\Desktop\ML-2\Stocks\Stocks\15_Min'

# # List all files in the directories
# indicators_files = [f for f in os.listdir(indicators_dir) if f.endswith('.xlsx')]
# fifteen_min_files = [f for f in os.listdir(fifteen_min_dir) if f.endswith('.xlsx')]

# # Ensure the number of files match
# if len(indicators_files) != len(fifteen_min_files):
#     raise ValueError("The number of Indicators files does not match the number of 15-Min files.")

# # Process each file pair
# for indicator_file in indicators_files:
#     # Get the corresponding fifteen-minute file
#     fifteen_min_file = indicator_file  # Assuming filenames are the same
#     if fifteen_min_file not in fifteen_min_files:
#         print(f"No corresponding file found for {indicator_file}")
#         continue

#     # Read the Indicators file
#     indicators_df = pd.read_excel(os.path.join(indicators_dir, indicator_file))

#     # Get the timestamp of the first entry
#     first_entry_timestamp = indicators_df.iloc[0]['DateTime']

#     # Read the 15-Min file
#     fifteen_min_df = pd.read_excel(os.path.join(fifteen_min_dir, fifteen_min_file))

#     # Convert 'datetime' column to datetime format for comparison
#     fifteen_min_df['datetime'] = pd.to_datetime(fifteen_min_df['datetime'])

#     # Convert the timestamp from the Indicators file to datetime format
#     first_entry_timestamp = pd.to_datetime(first_entry_timestamp)

#     # Find the row index where the timestamp matches
#     matching_row_index = fifteen_min_df[
#         abs(fifteen_min_df['datetime'] - first_entry_timestamp) < pd.Timedelta(seconds=1)
#     ].index

#     # Check if a match was found
#     if not matching_row_index.empty:
#         matching_row_index = matching_row_index[0]  # Get the first match
#         # Print the RSI value of the matched row
#         print(f"File: {indicator_file}")
#         print(f"RSI value of the matched row ({first_entry_timestamp}): {fifteen_min_df.loc[matching_row_index, 'RSI']}")

#         # Get the RSI values from the 2 cells below the matched row
#         if matching_row_index + 3 < len(fifteen_min_df):
#             rsi_values = fifteen_min_df.loc[matching_row_index + 1:matching_row_index + 3, 'RSI']
#             print(f"RSI values of the 3 cells below the matched row: {rsi_values.tolist()}")
#         else:
#             print("Not enough rows below the matched row to retrieve RSI values.")
#     else:
#         print("No matching timestamp found in the 15-Min file.")

#     print("\n" + "="*50 + "\n")


In [ ]:
import pandas as pd
import os

# Updated to use the Colab directories defined in cell 472b427f
indicators_dir = output_directory
fifteen_min_dir = input_directory

# List all .xlsx files in the directories
indicators_files = [f for f in os.listdir(indicators_dir) if f.endswith('.xlsx')]
fifteen_min_files = [f for f in os.listdir(fifteen_min_dir) if f.endswith('.xlsx')]

# Find the intersection of files (files that exist in both folders)
matching_files = set(indicators_files) & set(fifteen_min_files)

if not matching_files:
    print("No matching files found between Indicators and 15-Min folders.")
else:
    print(f"Processing {len(matching_files)} matching files...\n")

    # Process each matching file
    for file in matching_files:
        try:
            # Read the Indicators file
            indicators_df = pd.read_excel(os.path.join(indicators_dir, file))
            if indicators_df.empty:
                continue
            first_entry_timestamp = pd.to_datetime(indicators_df.iloc[0]['DateTime'])

            # Read the 15-Min file
            fifteen_min_df = pd.read_excel(os.path.join(fifteen_min_dir, file))
            fifteen_min_df['datetime'] = pd.to_datetime(fifteen_min_df['datetime'])

            # Find the row where timestamp matches (within 1 second tolerance)
            matching_row = fifteen_min_df[
                abs(fifteen_min_df['datetime'] - first_entry_timestamp) < pd.Timedelta(seconds=1)
            ]

            if not matching_row.empty:
                row_index = matching_row.index[0]
                print(f"File: {file}")
                print(f"RSI value of the matched row ({first_entry_timestamp}): {fifteen_min_df.loc[row_index, 'RSI']}")

                # Get the RSI values from the 3 rows below the matched row
                end_index = min(row_index + 3, len(fifteen_min_df) - 1)
                rsi_values = fifteen_min_df.loc[row_index + 1:end_index, 'RSI'].tolist()
                print(f"RSI values of the 3 rows below the matched row: {rsi_values}")
            else:
                print(f"No matching timestamp found in 15-Min file for {file}.")

            print("\n" + "="*50 + "\n")

        except Exception as e:
            print(f"Error processing {file}: {e}")

## RSI Comparision

In [ ]:
# import pandas as pd
# import os

# # Directory paths
# indicators_dir = r'C:\Users\adity\Desktop\ML-2\Stocks\Stocks\Indicators'
# fifteen_min_dir = r'C:\Users\adity\Desktop\ML-2\Stocks\Stocks\15_Min'

# # List all files in the directories
# indicators_files = [f for f in os.listdir(indicators_dir) if f.endswith('.xlsx')]
# fifteen_min_files = [f for f in os.listdir(fifteen_min_dir) if f.endswith('.xlsx')]

# # Ensure the number of files match
# if len(indicators_files) != len(fifteen_min_files):
#     raise ValueError("The number of Indicators files does not match the number of 15-Min files.")

# # Process each file pair
# for indicator_file in indicators_files:
#     # Get the corresponding fifteen-minute file
#     fifteen_min_file = indicator_file  # Assuming filenames are the same
#     if fifteen_min_file not in fifteen_min_files:
#         print(f"No corresponding file found for {indicator_file}")
#         continue

#     # Read the Indicators file
#     indicators_df = pd.read_excel(os.path.join(indicators_dir, indicator_file))

#     # Ensure the 'DateTime' column exists in the Indicators file
#     if 'DateTime' not in indicators_df.columns:
#         print(f"'DateTime' column not found in {indicator_file}")
#         continue

#     # Get the timestamp of the first entry
#     first_entry_timestamp = indicators_df.iloc[0]['DateTime']

#     # Read the 15-Min file
#     fifteen_min_df = pd.read_excel(os.path.join(fifteen_min_dir, fifteen_min_file))

#     # Standardize column names to lowercase and strip spaces
#     fifteen_min_df.columns = fifteen_min_df.columns.str.strip().str.lower()

#     # Ensure the 'datetime' column exists in the 15-Min file
#     if 'datetime' not in fifteen_min_df.columns:
#         print(f"'datetime' column not found in {fifteen_min_file}. Available columns: {fifteen_min_df.columns}")
#         continue

#     # Convert 'datetime' column to datetime format for comparison
#     fifteen_min_df['datetime'] = pd.to_datetime(fifteen_min_df['datetime'], errors='coerce')

#     # Convert the timestamp from the Indicators file to datetime format
#     first_entry_timestamp = pd.to_datetime(first_entry_timestamp, errors='coerce')

#     # Check if conversion is successful
#     if pd.isna(first_entry_timestamp):
#         print(f"Invalid timestamp in Indicators file: {indicator_file}")
#         continue

#     # Find the row index where the timestamp matches
#     matching_row_index = fifteen_min_df[fifteen_min_df['datetime'] == first_entry_timestamp].index

#     # Initialize variables
#     rsi_future_values = []
#     changes = []
#     trend = ""
#     rsi_level = ""

#     # Check if a match was found
#     if not matching_row_index.empty:
#         matching_row_index = matching_row_index[0]  # Get the first match

#         # Ensure the 'RSI' column exists
#         if 'rsi' not in fifteen_min_df.columns:
#             print(f"'RSI' column not found in {fifteen_min_file}")
#             continue

#         # Retrieve the RSI value of the matched row
#         rsi_current = fifteen_min_df.loc[matching_row_index, 'rsi']
#         print(f"File: {indicator_file}")
#         print(f"RSI value of the matched row ({first_entry_timestamp}): {rsi_current}")

#         # Get the RSI values from the 3 rows after the matched row
#         if matching_row_index + 3 < len(fifteen_min_df):
#             rsi_future_values = fifteen_min_df.loc[matching_row_index + 1:matching_row_index + 3, 'rsi']
#             rsi_future_values = rsi_future_values.tolist()

#             # Compute the change in RSI value
#             changes = [future_rsi - rsi_current for future_rsi in rsi_future_values]
#             print(f"RSI values of the next 3 rows: {rsi_future_values}")
#             print(f"Changes in RSI values compared to the current row: {changes}")

#             # Determine the trend (increasing or decreasing)
#             if all(change > 0 for change in changes):
#                 trend = "Increasing"
#             elif all(change < 0 for change in changes):
#                 trend = "Decreasing"
#             else:
#                 trend = "Mixed"
#             print(f"Trend: {trend}")
#         else:
#             print("Not enough rows after the matched row to retrieve RSI values.")

#         # Check RSI levels
#         if rsi_current >= 70:
#             rsi_level = "overbought"
#         elif rsi_current <= 30:
#             rsi_level = "oversold"
#         else:
#             rsi_level = "normal"
#         print(f"The RSI value is {rsi_current}. It is considered {rsi_level}.")

#         # Prepare data to be saved in the new sheet
#         rsi_results = {
#             'Timestamp': [first_entry_timestamp],
#             'Current RSI': [rsi_current],
#             'Future RSI Values': [rsi_future_values],
#             'Changes': [changes],
#             'Trend': [trend],
#             'RSI Level': [rsi_level]
#         }
#         rsi_results_df = pd.DataFrame(rsi_results)

#         # Save results to a new sheet in the same Indicators file
#         with pd.ExcelWriter(os.path.join(indicators_dir, indicator_file), engine='openpyxl', mode='a') as writer:
#             rsi_results_df.to_excel(writer, sheet_name='RSI Values', index=False)

#     else:
#         print(f"No matching timestamp found in the 15-Min file for {indicator_file}.")

#     print("\n" + "="*50 + "\n")


In [ ]:
import pandas as pd
import os

# Using Colab global variables: output_directory (Indicators) and input_directory (15_Min)
indicators_files = [f for f in os.listdir(output_directory) if f.endswith('.xlsx')]
fifteen_min_files = [f for f in os.listdir(input_directory) if f.endswith('.xlsx')]

matching_files = list(set(indicators_files) & set(fifteen_min_files))
print(f"Processing {len(matching_files)} matching files for RSI Comparison...")

for file in matching_files:
    try:
        indicators_df = pd.read_excel(os.path.join(output_directory, file))
        if 'DateTime' not in indicators_df.columns or len(indicators_df) == 0: continue
        first_entry_timestamp = pd.to_datetime(indicators_df.iloc[0]['DateTime'], errors='coerce')

        fifteen_min_df = pd.read_excel(os.path.join(input_directory, file))
        fifteen_min_df.columns = fifteen_min_df.columns.str.strip().str.lower()
        fifteen_min_df['datetime'] = pd.to_datetime(fifteen_min_df['datetime'], errors='coerce')

        matching_row_index = fifteen_min_df[fifteen_min_df['datetime'] == first_entry_timestamp].index
        if matching_row_index.empty: continue

        row_idx = matching_row_index[0]
        rsi_current = fifteen_min_df.loc[row_idx, 'rsi']
        rsi_future_values = fifteen_min_df.loc[row_idx + 1:row_idx + 3, 'rsi'].tolist()
        changes = [val - rsi_current for val in rsi_future_values]

        trend = "Mixed"
        if all(c > 0 for c in changes): trend = "Increasing"
        elif all(c < 0 for c in changes): trend = "Decreasing"

        rsi_level = "normal"
        if rsi_current >= 70: rsi_level = "overbought"
        elif rsi_current <= 30: rsi_level = "oversold"

        rsi_results_df = pd.DataFrame({
            'Timestamp': [first_entry_timestamp],
            'Current RSI': [rsi_current],
            'Future RSI Values': [rsi_future_values],
            'Changes': [changes],
            'Trend': [trend],
            'RSI Level': [rsi_level]
        })
        with pd.ExcelWriter(os.path.join(output_directory, file), engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            rsi_results_df.to_excel(writer, sheet_name='RSI Values', index=False)

    except Exception as e:
        print(f"Error in {file}: {e}")
print("RSI Comparison complete.")

## Writing to Analysis of Stock Excel

In [ ]:
import pandas as pd
import os

# Ensure search_date is defined (defaults to the value in the config cell if already run)
try:
    current_search_date = search_date
except NameError:
    current_search_date = '2026-04-15' # Fallback default

def read_and_filter_excel_files(input_dir, search_date_val, output_file):
    excel_files = [f for f in os.listdir(input_dir) if f.endswith('.xlsx')]
    if not excel_files:
        print('No files found to filter.')
        return

    matching_records_default = pd.DataFrame()
    matching_records_rsi = pd.DataFrame()

    for file in excel_files:
        file_path = os.path.join(input_dir, file)
        try:
            xl = pd.ExcelFile(file_path, engine='openpyxl')

            # Sheet 1 processing
            default_df = pd.read_excel(file_path, sheet_name=0, engine='openpyxl')
            if 'DateTime' in default_df.columns:
                default_df['DateTime'] = pd.to_datetime(default_df['DateTime'], errors='coerce')
                filtered = default_df[default_df['DateTime'].dt.date == pd.to_datetime(search_date_val).date()]
                if not filtered.empty:
                    filtered = filtered.copy()
                    filtered['SourceFile'] = file
                    matching_records_default = pd.concat([matching_records_default, filtered], ignore_index=True)

            # RSI Sheet processing
            if 'RSI Values' in xl.sheet_names:
                rsi_df = pd.read_excel(file_path, sheet_name='RSI Values', engine='openpyxl')
                if 'Timestamp' in rsi_df.columns:
                    rsi_df['Timestamp'] = pd.to_datetime(rsi_df['Timestamp'], errors='coerce')
                    filtered_rsi = rsi_df[rsi_df['Timestamp'].dt.date == pd.to_datetime(search_date_val).date()]
                    if not filtered_rsi.empty:
                        filtered_rsi = filtered_rsi.copy()
                        filtered_rsi['SourceFile'] = file
                        matching_records_rsi = pd.concat([matching_records_rsi, filtered_rsi], ignore_index=True)
        except Exception as e:
            print(f'Error processing {file}: {e}')

    if matching_records_default.empty and matching_records_rsi.empty:
        print(f'No records found for date {search_date_val}.')
        return

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        if not matching_records_default.empty:
            matching_records_default.to_excel(writer, sheet_name='Default Data', index=False)
        if not matching_records_rsi.empty:
            matching_records_rsi.to_excel(writer, sheet_name='RSI Values Data', index=False)
    print(f'Matching records saved to {output_file}')

read_and_filter_excel_files(output_directory, current_search_date, output_excel_file)

## Renaming and Removing the Data

In [ ]:
import pandas as pd
import os

def rename_source_file_column(input_file, output_file):
    if not os.path.exists(input_file):
        print(f'File {input_file} not found.')
        return

    sheets = pd.read_excel(input_file, sheet_name=None, engine='openpyxl')
    for sheet_name, df in sheets.items():
        if 'SourceFile' in df.columns:
            df['ShareName'] = df['SourceFile'].str.replace(r'_15min\\.xlsx$', '', regex=True)
            df.drop(columns=['SourceFile'], inplace=True)

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        for sheet_name, df in sheets.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    print(f'Renamed SourceFile to ShareName in {output_file}')

rename_source_file_column(output_excel_file, output_excel_file)

## Combining Sheets

In [ ]:
# Re-using logic with Colab output_excel_file variable
df1 = pd.read_excel(output_excel_file, sheet_name='Default Data')
df2 = pd.read_excel(output_excel_file, sheet_name='RSI Values Data')
combined_df = pd.concat([df1, df2], axis=1)
with pd.ExcelWriter(output_excel_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    combined_df.to_excel(writer, sheet_name='CombinedSheet', index=False)
print("Sheets combined.")

## Final Result

In [ ]:
# Re-using logic with Colab output_excel_file and final_output_file variables
data = pd.read_excel(output_excel_file, sheet_name='CombinedSheet')
data['ShareName'] = data['ShareName'].astype(str).str.strip()
output_data = data[['ShareName', 'DateTime', 'Close', 'Filter', 'Signal', 'Current RSI', 'Trend', 'RSI Level']]
output_data.to_excel(final_output_file, index=False)
print(f"Final result saved to {final_output_file}")

In [ ]:
import pandas as pd
import os

# Updated to use the variable final_output_file defined in cell 472b427f
input_file = final_output_file
output_file = final_output_file

if not os.path.exists(input_file):
    print(f"Error: {input_file} not found. Ensure the 'Final Result' cell above was executed.")
else:
    # Load the data from the sheet (Default sheet name is 'Sheet1' if not specified during to_excel)
    # Since the previous cell didn't specify a sheet name, it defaults to 'Sheet1'
    data = pd.read_excel(input_file)

    # Remove rows with any empty cells
    cleaned_data = data.dropna()

    # Write the cleaned data back to the same file
    cleaned_data.to_excel(output_file, index=False)

    print(f"Rows with empty cells have been removed and the cleaned data has been saved to '{output_file}'.")

## HuggingFace Model Fine-tuning

This section handles the fine-tuning of the `DeepSeek-R1-Distill-Llama-8B` model using the Unsloth library to optimize for intraday trading signals.

In [ ]:
%%capture
# Install Unsloth and dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.30" "trl<0.9.0" peft accelerate bitsandbytes

# Force a reload of the site-packages to ensure unsloth is found
import site
from importlib import reload
reload(site)

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

# NOTE: Ensure you are on a GPU runtime (Runtime > Change runtime type > T4 GPU)
if not torch.cuda.is_available():
    print("WARNING: GPU not detected. Unsloth requires a GPU to run. Please switch your runtime type to T4 GPU.")
else:
    print("GPU detected! Proceeding with model loading...")

max_seq_length = 2048
dtype = None # None for auto detection
load_in_4bit = True # Use 4bit quantization to reduce memory usage

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq = None,
)

### Dataset Preparation
We will convert the rule-based signals into a format the model can learn from.

In [ ]:
import pandas as pd
from datasets import Dataset
import os

# Point directly to the Colab path
path_to_load = "/content/Final_Result.xlsx"

if not os.path.exists(path_to_load):
    print(f"Error: {path_to_load} not found. Please run all processing cells first.")
else:
    train_df = pd.read_excel(path_to_load)

    # Filtering rows that have a valid Signal (Buy/Sell) to teach the model those patterns
    # We also include some 'Hold' or NaN cases if we want it to learn when NOT to trade
    train_df['Signal'] = train_df['Signal'].fillna('Hold')

    def format_instruction(row):
        return f"""Below is an intraday trading scenario. Analyze the indicators and provide the trading signal.

### Input:
Stock: {row['ShareName']}
Price: {row['Close']}
RSI: {row['Current RSI']}
Trend: {row['Trend']}
RSI Level: {row['RSI Level']}

### Response:
Based on the current RSI of {row['Current RSI']} and the {row['Trend']} trend, the AI signal is {row['Signal']}."""

    train_df['text'] = train_df.apply(format_instruction, axis=1)
    dataset = Dataset.from_pandas(train_df[['text']])
    print(f"Dataset prepared with {len(train_df)} rows for AI Signal Fine-tuning.")

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### 1. Pre-Training Check
Run this cell to ensure your GPU is active and the data file from the previous steps is ready.

In [ ]:
import torch
import os

# Check for GPU
if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected! Go to 'Runtime' > 'Change runtime type' and select 'T4 GPU'.")

# Check for Data
path_to_check = "/content/Final_Result.xlsx"
if not os.path.exists(path_to_check):
    raise FileNotFoundError(f"{path_to_check} not found. Please run the data processing cells above to generate it.")

print("✅ GPU detected and Data File found. Ready for training.")

### 2. Execute Fine-Tuning
This will start the training process using the parameters optimized for Colab.

In [ ]:
import torch
# Verify model and trainer are loaded from previous cells
try:
    trainer_stats = trainer.train()
    print("Training Complete!")
except NameError:
    print("Error: 'trainer' or 'model' not defined. Please ensure the 'HuggingFace Model Fine-tuning' cells were run successfully.")

In [ ]:
import pandas as pd
import os
from unsloth import FastLanguageModel

# 1. Setup paths
input_dir = '/content/15_Min/'
analysis_file = "/content/Analysis_of_Stocks.xlsx"

# 2. Set model to inference mode
FastLanguageModel.for_inference(model)

def get_ai_signal(row):
    prompt = f"""Below is an intraday trading scenario. Analyze the indicators and provide the trading signal.

### Input:
Stock: {row.get('ShareName', 'Unknown')}
Price: {row.get('Close', 0)}
RSI: {row.get('RSI', 0)}
Trend: {row.get('Trend', 'Unknown')}
RSI Level: {row.get('RSI Level', 'Unknown')}

### Response:
Based on the current RSI"""

    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 64)
    response = tokenizer.batch_decode(outputs)[0]

    if "AI signal is Buy" in response: return "AI Buy"
    elif "AI signal is Sell" in response: return "AI Sell"
    else: return "AI Hold"

# 3. Process all files in the 15_Min directory
all_ai_results = []
files_to_analyze = [f for f in os.listdir(input_dir) if f.endswith('.xlsx')]

print(f"Analyzing {len(files_to_analyze)} files from {input_dir} using AI model...")

for file in files_to_analyze:
    try:
        df = pd.read_excel(os.path.join(input_dir, file))
        # Basic cleaning for the prompt context
        df.columns = [c.strip() for c in df.columns]
        share_name = file.replace('_15min.xlsx', '')

        # Just take the last row or relevant rows for signal generation
        # Here we process the latest data point for each stock
        latest_row = df.iloc[-1].to_dict()
        latest_row['ShareName'] = share_name

        # Ensure we have a dummy trend/level if not in the raw 15min file yet
        if 'RSI' not in latest_row: latest_row['RSI'] = 0
        latest_row['Trend'] = 'Unknown' # AI will determine or use logic
        latest_row['RSI Level'] = 'Normal'

        signal = get_ai_signal(latest_row)
        all_ai_results.append({
            'ShareName': share_name,
            'DateTime': latest_row.get('datetime', latest_row.get('DateTime')),
            'AI_Generated_Signal': signal
        })
    except Exception as e:
        print(f"Error processing {file}: {e}")

# 4. Save results
ai_df = pd.DataFrame(all_ai_results)
if os.path.exists(analysis_file):
    with pd.ExcelWriter(analysis_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        ai_df.to_excel(writer, sheet_name='AI Signals', index=False)
    print(f"Success! AI Signals added to {analysis_file} based on raw directory analysis.")
else:
    ai_df.to_excel(analysis_file, index=False)
    print(f"Success! Created {analysis_file} with AI Signals.")